# ULTRA-MoCap LOSO Training on Colab GPU

This notebook is an orchestrator. The full training logic is in:
Code-base/MocapDatasetScripting_REALLAB/scripts/training/conv1d_bigru_loso.py

Why the cells are small:
- they only set runtime, paths, and environment variables
- they launch the full script unchanged
- this keeps one source of truth for training behavior

Run order:
1. Connect runtime with GPU
2. Run config cell (auto-resolves repo + dataset paths)
3. Run launch cell (starts full 13-fold LOSO)
4. Run sync cell (copies cache results back to Drive)

In [1]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)

In [3]:
%pip install -q numpy pandas scipy scikit-learn tqdm h5py

In [6]:
from pathlib import Path
import subprocess

# ---------------- Explicit path mode (default) ----------------
# No recursive Drive search in this mode.
# Set these to the exact locations you want.

# Local workspace defaults.
REPO_DIR = '/Users/meghvyas/Desktop/research-paper'
H5_PATH = '/Users/meghvyas/Desktop/research-paper/Dataset/ULTra-MoCap-processed/All_subjects_data.h5'

# Colab repo alternative if you choose to run in Colab.
REPO_CANDIDATES = [
    '/Users/meghvyas/Desktop/research-paper',
    '/content/ULTRA-MoCap-Kinematics-Analysis',
]

# Default H5 candidates are local-only to avoid accidental Drive selection.
H5_CANDIDATES = [
    '/Users/meghvyas/Desktop/research-paper/Dataset/ULTra-MoCap-processed/All_subjects_data.h5',
    '/Users/meghvyas/Desktop/research-paper/Dataset/28751156/ULTra-MoCap-processed/All_subjects_data.h5',
    '/content/ULTRA-MoCap-Kinematics-Analysis/Dataset/ULTra-MoCap-processed/All_subjects_data.h5',
    '/content/ULTRA-MoCap-Kinematics-Analysis/Dataset/28751156/ULTra-MoCap-processed/All_subjects_data.h5',
]

# Optional and OFF by default: set True only if you explicitly want Drive-wide search.
ALLOW_DRIVE_SCAN = False

# Resolve repository from explicit candidates only.
repo_paths = [Path(p) for p in ([REPO_DIR] + REPO_CANDIDATES)]
_repo = next((p for p in repo_paths if p.exists()), None)

if _repo is None:
    _repo = Path('/content/ULTRA-MoCap-Kinematics-Analysis')
    print('Cloning repo to /content...')
    subprocess.run([
        'git', 'clone',
        'https://github.com/MeghVyas3132/ULTRA-MoCap-Kinematics-Analysis.git',
        str(_repo),
    ], check=True)

REPO_DIR = str(_repo)

# Resolve H5 strictly from explicit path + ordered candidates.
checked_h5_paths = []
for p in [H5_PATH] + H5_CANDIDATES:
    pp = Path(p)
    if pp not in checked_h5_paths:
        checked_h5_paths.append(pp)

_h5 = next((p for p in checked_h5_paths if p.exists()), None)

if _h5 is None and ALLOW_DRIVE_SCAN:
    drive_roots = [Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')]
    drive_root = next((p for p in drive_roots if p.exists()), None)
    if drive_root is not None:
        hits = list(drive_root.rglob('All_subjects_data.h5'))
        if hits:
            _h5 = hits[0]

if _h5 is None:
    print('H5 not found. Checked explicit paths in order:')
    for p in checked_h5_paths:
        print('  -', p)
    print('Recursive Drive search is disabled (ALLOW_DRIVE_SCAN=False).')
    raise RuntimeError(
        'All_subjects_data.h5 not found. Set H5_PATH to the exact file path or update H5_CANDIDATES.'
    )

H5_PATH = str(_h5)

# Training run config.
USE_LOCAL_SSD_CACHE = True
if USE_LOCAL_SSD_CACHE:
    DATASET_ROOT = '/content/mocap_cache/datasets' if REPO_DIR.startswith('/content/') else f"{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/datasets"
    RESULTS_FOLDER = '/content/mocap_cache/results/Results_ConvBiGRU_colab' if REPO_DIR.startswith('/content/') else f"{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/results/Results_ConvBiGRU_colab"
else:
    DATASET_ROOT = f"{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/datasets"
    RESULTS_FOLDER = f"{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/results/Results_ConvBiGRU_colab"

RESULTS_ZIP_PATH = f"{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/results/ConvBiGRU_results_colab_gpu.zip"

# Best-effort Drive sync target; falls back to local results folder when Drive is unavailable.
_default_drive_results = Path('/content/drive/MyDrive/research-paper/Code-base/MocapDatasetScripting_REALLAB/results/Results_ConvBiGRU_colab_gpu')
DRIVE_RESULTS_DIR = str(_default_drive_results if _default_drive_results.parent.exists() else Path(RESULTS_FOLDER))

# Full LOSO: 0 means all 13 folds.
MAX_FOLDS = 0
MODALITIES = 'emg,imu,imu_emg'
RESULT_TAG = 'colab_full_loso'

# EMG-specific overrides.
EMG_MODEL_VARIANT = 'lstm_msa'
EMG_EPOCHS = 50
EMG_PATIENCE = 12
EMG_LR = 5e-4

print('REPO_DIR =', REPO_DIR)
print('H5_PATH =', H5_PATH)
print('ALLOW_DRIVE_SCAN =', ALLOW_DRIVE_SCAN)
print('DATASET_ROOT =', DATASET_ROOT)
print('RESULTS_FOLDER =', RESULTS_FOLDER)
print('DRIVE_RESULTS_DIR =', DRIVE_RESULTS_DIR)
print('MAX_FOLDS =', MAX_FOLDS)
print('MODALITIES =', MODALITIES)
print('RESULT_TAG =', RESULT_TAG)

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), (
    'CUDA GPU is not available in this runtime. '
    'Switch Colab runtime to GPU and rerun preflight.'
)
print('Launching on GPU:', torch.cuda.get_device_name(0))

assert Path(REPO_DIR).exists(), f'Repo path not found: {REPO_DIR}'
assert Path(H5_PATH).exists(), f'H5 path not found: {H5_PATH}'

Path(DATASET_ROOT).mkdir(parents=True, exist_ok=True)
Path(RESULTS_FOLDER).mkdir(parents=True, exist_ok=True)
Path(RESULTS_ZIP_PATH).parent.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env.update({
    'FAST_MODE': '1',
    'MATMUL_PRECISION': 'high',
    'DATA_H5_PATH': H5_PATH,
    'DATASET_ROOT': DATASET_ROOT,
    'RESULTS_FOLDER': RESULTS_FOLDER,
    'RESULTS_ZIP_PATH': RESULTS_ZIP_PATH,
    'RESULT_TAG': RESULT_TAG,
    'MAX_FOLDS': str(MAX_FOLDS),
    'MODALITIES': MODALITIES,
    'DATALOADER_WORKERS': '8',
    'PREFETCH_FACTOR': '4',
    'PERSISTENT_WORKERS': '1',
    'EMG_MODEL_VARIANT': EMG_MODEL_VARIANT,
    'EMG_EPOCHS': str(EMG_EPOCHS),
    'EMG_PATIENCE': str(EMG_PATIENCE),
    'EMG_LR': str(EMG_LR),
})

cmd = [
    sys.executable,
    '-u',
    'Code-base/MocapDatasetScripting_REALLAB/scripts/training/conv1d_bigru_loso.py',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)

In [ ]:
import shutil
from pathlib import Path

Path(DRIVE_RESULTS_DIR).mkdir(parents=True, exist_ok=True)
if Path(RESULTS_FOLDER).resolve() != Path(DRIVE_RESULTS_DIR).resolve():
    shutil.copytree(RESULTS_FOLDER, DRIVE_RESULTS_DIR, dirs_exist_ok=True)

print('Synced results dir:', DRIVE_RESULTS_DIR)
print('Zip path:', RESULTS_ZIP_PATH)